<a href="https://colab.research.google.com/github/03sarath/vertex-ai-mlops-kfp2/blob/main/Vertex_AI_kfp2_pipelinep3-11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Copyright & License (click to expand)



In [ ]:
# @title Copyright & License (click to expand)
# Copyright 2023 Psitron Technologies Pvt Ltd
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## Overview

This tutorial uses the Vertex AI Pipelines with KFP 2.x.


### Objective: **🍷** **Predicting wine quality**

In this tutorial, you learn to use `Vertex AI Pipelines` and KFP 2.x version of `Google Cloud Pipeline Components` to train and deploy an RandomForestClassifier model.


This tutorial uses the following Google Cloud ML services:

- `Vertex AI Pipelines`
- `Google Cloud Pipeline Components`

The steps performed include:

- Create a KFP pipeline:
    - Data preprocessing.
    - Train an RandomForestClassifier `Model`.
    - Evaluation an `Model`.
    - Create an `Endpoint` resource.
    - Deploys the `Model` resource to the `Endpoint` resource.
- Compile the KFP pipeline.
- Execute the KFP pipeline using `Vertex AI Pipelines`

The components are [documented here](https://google-cloud-pipeline-components.readthedocs.io/en/google-cloud-pipeline-components-1.0.0/google_cloud_pipeline_components.aiplatform.html).

### Install Vertex AI SDK for Python and other required packages

In [ ]:
! pip3 install --no-cache-dir --upgrade "kfp>2" \
                                        google-cloud-aiplatform

Check the KFP SDK version.

In [ ]:

! python3 -c "import kfp; print('KFP SDK version: {}'.format(kfp.__version__))"
! pip3 freeze | grep aiplatform

### Restart runtime (Colab only)

To use the newly installed packages, you must restart the runtime on Google Colab.

In [ ]:
import sys

if "google.colab" in sys.modules:

    import IPython

    app = IPython.Application.instance()
    app.kernel.do_shutdown(True)

### Authenticate your notebook environment (Colab only)

Authenticate your environment on Google Colab.

In [ ]:

import sys

if "google.colab" in sys.modules:

    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
# DEFINE THESE VARIABLES AT THE TOP
PROJECT_ID = "mlopstt"  # Your project ID
LOCATION = "us-central1"  # Your region
PIPELINE_ROOT = f"gs://{PROJECT_ID}-pipeline-root"  # Your GCS bucket

### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

In [ ]:
BUCKET_URI = f"gs://your-bucket-name-{PROJECT_ID}-unique"  # @param {type:"string"}


**Only if your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [ ]:
! gsutil mb -l $LOCATION -p $PROJECT_ID $BUCKET_URI

#### Service Account

You use a service account to create Vertex AI Pipeline jobs.

In [ ]:
SERVICE_ACCOUNT = "[your-service-account]"  # @param {type:"string"}

In [ ]:
import sys

IS_COLAB = "google.colab" in sys.modules
if (
    SERVICE_ACCOUNT == ""
    or SERVICE_ACCOUNT is None
    or SERVICE_ACCOUNT == "[your-service-account]"
):
    if not IS_COLAB:
        shell_output = !gcloud auth list 2>/dev/null
        SERVICE_ACCOUNT = shell_output[2].replace("*", "").strip()

    else:  # IS_COLAB:
        shell_output = !gcloud projects describe $PROJECT_ID

        # Debug: print output to see what we're working with
        print("gcloud output:", shell_output)

        # Find the line containing 'projectNumber' instead of assuming it's last
        project_number = None
        for line in shell_output:
            if "projectNumber" in line and ":" in line:
                project_number = line.split(":", 1)[1].strip().replace("'", "")
                break

        if not project_number:
            raise ValueError(
                f"Could not find projectNumber in gcloud output. "
                f"Make sure PROJECT_ID='{PROJECT_ID}' is correct and you're authenticated."
            )

        SERVICE_ACCOUNT = f"{project_number}-compute@developer.gserviceaccount.com"

    print("Service Account:", SERVICE_ACCOUNT)

#### Set service account access for Vertex AI Pipelines

Run the following commands to grant your service account access to read and write pipeline artifacts in the bucket that you created in the previous step. You only need to run this step once per service account.

In [ ]:
! gsutil iam ch serviceAccount:{SERVICE_ACCOUNT}:roles/storage.objectCreator $BUCKET_URI

! gsutil iam ch serviceAccount:{SERVICE_ACCOUNT}:roles/storage.objectViewer $BUCKET_URI

### Import libraries and define constants

In [ ]:
import google.cloud.aiplatform as aiplatform
import kfp
from kfp import compiler, dsl
from kfp.dsl import Artifact, Dataset, Input, Metrics, Model, Output, component, ClassificationMetrics
from typing import NamedTuple
from google.cloud.aiplatform import pipeline_jobs
from datetime import datetime

## Initialize Vertex AI SDK for Python

Initialize the Vertex AI SDK for Python for your project and corresponding bucket.

In [ ]:
aiplatform.init(project=PROJECT_ID, staging_bucket=BUCKET_URI)

### Define Data preparation component


In [ ]:
@component(
    packages_to_install=["numpy==1.24.3", "pandas==2.2.2", "pyarrow", "scikit-learn==1.4.2"],
)
def get_wine_data(
    url: str,
    dataset_train: Output[Dataset],
    dataset_test: Output[Dataset]
):
    import pandas as pd
    import numpy as np
    import os
    from sklearn.model_selection import train_test_split as tts

    df_wine = pd.read_csv(url, delimiter=";")
    df_wine['best_quality'] = [1 if x >= 7 else 0 for x in df_wine.quality]
    df_wine['target'] = df_wine.best_quality
    df_wine = df_wine.drop(['quality', 'total sulfur dioxide', 'best_quality'], axis=1)

    train, test = tts(df_wine, test_size=0.3, random_state=42)

    train.to_csv(dataset_train.path, index=False, encoding='utf-8-sig')
    test.to_csv(dataset_test.path, index=False, encoding='utf-8-sig')

    print(f"Train path: {dataset_train.path}")
    print(f"Train exists: {os.path.exists(dataset_train.path)}, size: {os.path.getsize(dataset_train.path)} bytes")
    print(f"Test path: {dataset_test.path}")
    print(f"Test exists: {os.path.exists(dataset_test.path)}, size: {os.path.getsize(dataset_test.path)} bytes")

### Define RandomForestClassifier Model training component


In [ ]:
@component(
    packages_to_install=["numpy==1.24.3", "pandas==2.2.2", "scikit-learn==1.4.2"]
)
def train_winequality(
    dataset: Input[Dataset],
    model: Output[Model],
):
    from sklearn.ensemble import RandomForestClassifier
    import pandas as pd
    import joblib
    import os

    # FUSE read — standard KFP v2 pattern for sequential tasks
    data = pd.read_csv(dataset.path)
    print(f"Loaded dataset shape: {data.shape}")

    model_rf = RandomForestClassifier(n_estimators=10, random_state=42)
    model_rf.fit(data.drop(columns=["target"]), data.target)

    model.metadata["framework"] = "RF"

    # Save as model.joblib inside a directory — sklearn prebuilt container requirement
    os.makedirs(model.path, exist_ok=True)
    model_file = model.path + "/model.joblib"
    joblib.dump(model_rf, model_file)
    print(f"Model saved to: {model_file}")
    print(f"Model exists: {os.path.exists(model_file)}, size: {os.path.getsize(model_file)} bytes")

### Define Model evaluation component


In [ ]:
@component(
    packages_to_install=["numpy==1.24.3", "pandas==2.2.2", "scikit-learn==1.4.2"]
)
def winequality_evaluation(
    test_set: Input[Dataset],
    rf_winequality_model: Input[Model],
    thresholds_dict_str: str,
    metrics: Output[ClassificationMetrics],
    kpi: Output[Metrics]
) -> NamedTuple("output", [("deploy", str)]):
    import pandas as pd
    import joblib
    import json
    import numpy as np
    from sklearn.metrics import roc_curve, confusion_matrix, accuracy_score

    def threshold_check(val1, val2):
        return "true" if val1 >= val2 else "false"

    # FUSE read — standard KFP v2 pattern
    print("Loading test data...")
    data = pd.read_csv(test_set.path)
    print(f"Test data shape: {data.shape}")

    print("Loading model...")
    model = joblib.load(rf_winequality_model.path + "/model.joblib")

    print("Making predictions...")
    y_test = data.drop(columns=["target"])
    y_target = data["target"].astype(int)
    y_pred = model.predict(y_test)

    print("Calculating ROC curve...")
    y_scores = model.predict_proba(y_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(
        y_true=y_target.values,
        y_score=y_scores,
        pos_label=1
    )

    fpr_clean = [float(x) for x in fpr if not (np.isinf(x) or np.isnan(x))]
    tpr_clean = [float(x) for x in tpr if not (np.isinf(x) or np.isnan(x))]
    thresholds_clean = [
        float(t) if not (np.isinf(t) or np.isnan(t)) else 0.0
        for t in thresholds
    ]
    min_len = min(len(fpr_clean), len(tpr_clean), len(thresholds_clean))
    fpr_clean = fpr_clean[:min_len]
    tpr_clean = tpr_clean[:min_len]
    thresholds_clean = thresholds_clean[:min_len]

    try:
        metrics.log_roc_curve(fpr=fpr_clean, tpr=tpr_clean, thresholds=thresholds_clean)
        print("ROC curve logged")
    except Exception as e:
        print(f"Warning: ROC curve failed: {e}")

    cm = confusion_matrix(y_target, y_pred)
    cm_list = [[int(x) for x in row] for row in cm]
    try:
        metrics.log_confusion_matrix(["False", "True"], cm_list)
        print("Confusion matrix logged")
    except Exception as e:
        print(f"Warning: Confusion matrix failed: {e}")

    accuracy = float(accuracy_score(y_target, y_pred))
    print(f"Accuracy: {accuracy:.4f}")

    thresholds_dict = json.loads(thresholds_dict_str)
    threshold_value = float(thresholds_dict["roc"])

    rf_winequality_model.metadata["accuracy"] = accuracy
    kpi.log_metric("accuracy", accuracy)

    deploy = threshold_check(accuracy, threshold_value)
    print(f"Deploy decision: {deploy} (accuracy {accuracy:.4f} vs threshold {threshold_value})")
    return (deploy,)

### Define deploying the model component

Finally, you define a component to deploy the model.

In [ ]:
@component(
    packages_to_install=["google-cloud-aiplatform", "google-cloud-storage", "scikit-learn==1.4.2", "kfp", "numpy==1.24.3"],
    output_component_file="model_winequality_component.yml"
)
def deploy_winequality(
    model: Input[Model],
    project: str,
    region: str,
    serving_container_image_uri: str,
    vertex_endpoint: Output[Artifact],
    vertex_model: Output[Model]
):
    from google.cloud import aiplatform
    from google.cloud import storage
    import time

    aiplatform.init(project=project, location=region)

    DISPLAY_NAME = "winequality"
    ENDPOINT_NAME = "winequality_endpoint"

    print(f"Model URI: {model.uri}")

    def create_endpoint():
        endpoints = aiplatform.Endpoint.list(
            filter='display_name="{}"'.format(ENDPOINT_NAME),
            order_by='create_time desc',
            project=project,
            location=region,
        )
        if len(endpoints) > 0:
            endpoint = endpoints[0]
            print(f"Using existing endpoint: {endpoint.resource_name}")
        else:
            endpoint = aiplatform.Endpoint.create(
                display_name=ENDPOINT_NAME,
                project=project,
                location=region
            )
            print(f"Created new endpoint: {endpoint.resource_name}")
        return endpoint

    endpoint = create_endpoint()

    storage_client = storage.Client(project=project)

    # Download model.joblib from source via GCS client
    # (deploy runs on a different machine — no FUSE mount available)
    src_path = (model.uri.rstrip('/') + '/model.joblib').replace('gs://', '')
    src_bucket = src_path.split('/')[0]
    src_blob = '/'.join(src_path.split('/')[1:])
    print(f"Downloading model from: gs://{src_path}")
    model_bytes = storage_client.bucket(src_bucket).blob(src_blob).download_as_bytes()
    print("Model downloaded successfully")

    # Upload model.joblib to a clean destination directory
    timestamp = str(int(time.time()))
    dest_dir = model.uri.rstrip('/') + f'/serving_{timestamp}'
    dest_gcs_path = dest_dir.replace('gs://', '')
    dest_bucket = dest_gcs_path.split('/')[0]
    dest_blob = '/'.join(dest_gcs_path.split('/')[1:]) + '/model.joblib'
    storage_client.bucket(dest_bucket).blob(dest_blob).upload_from_string(model_bytes)
    print(f"Model uploaded to: gs://{dest_bucket}/{dest_blob}")

    # Upload to Vertex AI Model Registry
    # No health/predict routes needed — sklearn prebuilt container handles those internally
    print("Uploading model to Vertex AI Model Registry...")
    model_upload = aiplatform.Model.upload(
        display_name=DISPLAY_NAME,
        artifact_uri=dest_dir,
        serving_container_image_uri=serving_container_image_uri,
    )
    print(f"Model registered: {model_upload.resource_name}")

    # Deploy to endpoint
    print("Deploying model to endpoint...")
    model_upload.deploy(
        machine_type="n1-standard-4",
        endpoint=endpoint,
        traffic_split={"0": 100},
        deployed_model_display_name=DISPLAY_NAME,
    )
    print("Deployment complete")

    # model_upload is the Model Registry entry
    vertex_model.uri = model_upload.resource_name
    vertex_endpoint.uri = endpoint.resource_name

In [ ]:
from datetime import datetime

TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")
DISPLAY_NAME = 'pipeline-winequality-job{}'.format(TIMESTAMP)

## Construct the RandomForestClassifier training pipeline

Now you define the pipeline, with the following steps:

- Data preparation.
- Train the model.
- Evaluate the model.
- Deploy the model.

In [ ]:
@dsl.pipeline(
    pipeline_root=PIPELINE_ROOT,
    name="pipeline-winequality",
)
def pipeline(
    url: str = "https://psitron.s3.ap-southeast-1.amazonaws.com/vertexai/winequality-white.csv",
    project: str = PROJECT_ID,
    region: str = LOCATION,
    thresholds_dict_str: str = '{"roc":0.8}',
    serving_container_image_uri: str = "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-4:latest"
):
    data_op = get_wine_data(url=url)

    train_model_op = train_winequality(dataset=data_op.outputs["dataset_train"])

    model_evaluation_op = winequality_evaluation(
        test_set=data_op.outputs["dataset_test"],
        rf_winequality_model=train_model_op.outputs["model"],
        thresholds_dict_str=thresholds_dict_str,
    )

    with dsl.If(
        model_evaluation_op.outputs["deploy"] == "true",
        name="deploy-winequality",
    ):
        deploy_model_op = deploy_winequality(
            model=train_model_op.outputs['model'],
            project=project,
            region=region,
            serving_container_image_uri=serving_container_image_uri,
        )


### Compile the pipeline

Next, you compile the pipeline.

In [ ]:
# Compile
compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path='ml_winequality.json'
)

In [ ]:
# # Compile the pipeline
# compiler.Compiler().compile(
#     pipeline_func=pipeline,
#     package_path='ml_winequality.yaml'
# )

### Run the pipeline using Vertex AI Pipelines

Now, run the compiled pipeline.

In [ ]:
# Run
start_pipeline = pipeline_jobs.PipelineJob(
    display_name=DISPLAY_NAME,
    template_path="ml_winequality.json",
    enable_caching=False,
    location=LOCATION,
    project=PROJECT_ID,
)

In [ ]:
start_pipeline.run()

# ⏰ **Schedule** your pipelines to run recurrently

In [ ]:
#Schedule your pipelines to run recurrently

from google.cloud import aiplatform

pipeline_job = aiplatform.PipelineJob(
  template_path="ml_winequality.json",
  pipeline_root=PIPELINE_ROOT,
  display_name="ml_winequality",
)

pipeline_job_schedule = aiplatform.PipelineJobSchedule(
  pipeline_job=pipeline_job,
  display_name="ml_winequality"
)

pipeline_job_schedule.create(
  #cron="0 0 * * *",
  cron="TZ=Asia/Calcutta 0 0 * * *",
  max_concurrent_run_count=1,
  max_run_count=10,
)

#⏰List your all scheduled pipelines


In [ ]:
#List your all scheduled pipelines
from google.cloud import aiplatform

aiplatform.PipelineJobSchedule.list(
  filter='display_name="ml_winequality"',
  order_by='create_time desc'
)

#Invoke **ML model deployed in vertex ai endpoint**

In [ ]:
import subprocess
from google.cloud import aiplatform

ENDPOINT_NAME = "winequality_endpoint"

#Bad Qty Wine Data 👎
instance = [[7.2, 0.23, 0.32, 8.5, 0.058, 47, 132, 0.993, 3.12, 0.46]]

#Good Qty Wine Data 👍
#instance = [[6.7,0.23,0.31,2.1,0.046,30,0.9926,3.33,0.64,10.7]]


def endpoint_predict(project: str, location: str, instances: list, endpoint: str):
    aiplatform.init(project=project, location=location)
    endpoint = aiplatform.Endpoint(endpoint)
    prediction = endpoint.predict(instances=instances)
    return prediction

#prediction = endpoint_predict('<project_id>', 'us-central1', instance, '<endpoint_id>')
prediction = endpoint_predict('mlops-end-to-end-490913', 'us-central1', instance, '4347768593144872960')
print("Prediction from Deployed model_id:", prediction[1])
print("Prediction from Deployed model version_id:", prediction[3])
print("Predicted wine quality 🍷👍👎:", prediction[0])